# 2. Inference

Run `01_train.ipynb` first, then run this in a **fresh runtime** on a T4 GPU with *Runtime > Run all*.
Writes the test predictions, the submission file and the metrics.

In [ ]:
%pip install -q "transformers==5.22.2" "peft==0.21.2" accelerate sentencepiece protobuf pyarrow rouge-score sacrebleu bert-score
%pip uninstall -y torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# True runs a quick check with 300 rows. Use the same value in both notebooks.
SMOKE_TEST = False

import json
import sys
import time

import pandas as pd
from google.colab import drive

NOTEBOOK_START = time.time()
drive.mount("/content/drive")
sys.dont_write_bytecode = True
sys.path.insert(0, "/content/drive/MyDrive/agricultural_qa")

import agri_common as ac

ac.require_gpu()
paths = ac.Paths(SMOKE_TEST)

Mounted at /content/drive
GPU: Tesla T4
Run folder: /content/drive/MyDrive/agricultural_qa/fast_v1/2effa812


## Load data

In [ ]:
with ac.stage("Load data", paths):
    data = ac.load_data(paths, SMOKE_TEST)
train_df, test_df, unseen_df = data["train"], data["test"], data["unseen"]
if SMOKE_TEST:
    test_df = test_df.head(20).reset_index(drop=True)
print(f"test {len(test_df)} | unseen {len(unseen_df)}")
print("Unseen questions that also appear word for word in the training file:", int(unseen_df["in_training_file"].sum()))


--- Load data ---
[Load data] 4.2 s | GPU memory in use: 0.00 GB
test 1439 | unseen 1000
Unseen questions that also appear word for word in the training file: 435


## Retrieve context for every question

In [ ]:
with ac.stage("Retrieve", paths):
    encoder = ac.Encoder()
    retriever = ac.Retriever(encoder, paths)
    test_hits = retriever.search(test_df["question"])
    unseen_hits = retriever.search(unseen_df["question"])
    baseline_hits = retriever.search(test_df["question"], top_k=1, hybrid=False)
    test_prompts = [ac.make_prompt(q, c) for q, c in zip(test_df["question"], retriever.contexts(test_hits))]
    unseen_prompts = [ac.make_prompt(q, c) for q, c in zip(unseen_df["question"], retriever.contexts(unseen_hits))]
    retriever.encoder = None
    del encoder
    ac.free_gpu()


--- Retrieve ---


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

... (9 outputs left out)

Search:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding:   0%|          | 0/6 [00:00<?, ?it/s]

Search:   0%|          | 0/6 [00:00<?, ?it/s]

[Retrieve] 50.2 s | GPU memory in use: 0.01 GB


## Generate answers and save predictions

In [ ]:
with ac.stage("Generate", paths):
    tokenizer, model = ac.load_model(paths)
    test_answers = ac.generate_answers(tokenizer, model, test_prompts, "Test questions")
    ac.predictions_table(retriever, test_df, test_answers, test_hits).to_csv(
        paths.outputs / "test_predictions.csv", index=False, encoding="utf-8-sig")

    unseen_answers = ac.generate_answers(tokenizer, model, unseen_prompts, "Unseen questions")
    submission = ac.predictions_table(retriever, unseen_df, unseen_answers, unseen_hits)
    expected = pd.read_csv(paths.unseen_csv, dtype=str, keep_default_na=False)["question_id"].head(len(unseen_df)).tolist()
    if submission["question_id"].tolist() != expected or (not SMOKE_TEST and len(submission) != 1000):
        raise AssertionError("The submission doesn't match unseen_set_1000.csv")
    submission.to_csv(paths.outputs / "submission_predictions.csv", index=False, encoding="utf-8-sig")

    del model
    ac.free_gpu()

print("Saved predictions to", paths.outputs)
display(submission.head(3))


--- Generate ---


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (8 outputs left out)

Test questions: 8.78 questions per second


Unseen questions:   0%|          | 0/32 [00:00<?, ?it/s]

Unseen questions: 8.93 questions per second
[Generate] 284.9 s | GPU memory in use: 0.01 GB
Saved predictions to /content/drive/MyDrive/agricultural_qa/fast_v1/2effa812/outputs


,question_id,question,predicted_answer,predicted_sources,matched_image_files,detected_language,retrieval_scores,confidence
0,69a110f8993748a05837af6e,How can cotton farmers prepare homemade organi...,Cotton farmers can prepare homemade organic in...,"[""Package of Pratices Kharif - Haryana"", ""TNAU...",[],en,"[0.93, 0.9193, 0.9193]",0.9300
1,6a42401487c952ead5313665,What are the effective management strategies f...,Leeches are a common pest in papaya. They are ...,"[""PAPAYA POP"", ""AESA BASED IPM PACKAGE OF PAPA...",[],en,"[0.8697, 0.8644, 0.864]",0.8697
2,69e193725b44aebc5f16149e,How to cultivate Jhona (millet/sorghum) in Kat...,Jhona (millet/sorghum) is a cultivated rice va...,"[""Kharif POP-SKUAST_Jammu"", ""Jammu kasmir POP""...",[],en,"[0.8802, 0.8726, 0.8754]",0.8802


## Metrics

Baseline is the full training answer of the top embedding match. Final is the T5 answer.

In [ ]:
with ac.stage("Metrics", paths):
    answers_by_id = train_df.set_index("question_id")
    baseline_ids = [retriever.records(h, 1)[0] for h in baseline_hits]
    baseline_answers = [answers_by_id.at[q, "final_answer"] for q in baseline_ids]
    baseline_sources = [json.loads(answers_by_id.at[q, "sources"]) for q in baseline_ids]
    final_sources = [retriever.source_names(h) for h in test_hits]

    results = pd.DataFrame(ac.evaluate_answers("Baseline", baseline_answers, baseline_sources, test_df)
                           + ac.evaluate_answers("Final", test_answers, final_sources, test_df))
    results.to_csv(paths.outputs / "evaluation_results.csv", index=False)

display(results.round(4))


--- Metrics ---


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (3 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Metrics] 341.4 s | GPU memory in use: 0.01 GB


,model,language,n,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore-F1,number_recall,dosage_recall,source_hit_rate,language_matches_question,SacreBLEU
0,Baseline,overall,1439,0.3918,0.1471,0.2088,0.7194,0.4604,0.2565,0.1084,0.9965,12.1815
1,Baseline,en,1436,0.3922,0.1473,0.2091,0.7195,0.4601,0.2569,0.1086,0.9986,12.2222
2,Baseline,pa,3,0.1924,0.0337,0.0885,0.6642,0.5927,0.0000,0.0000,0.0000,1.3009
3,Final,overall,1439,0.2350,0.0688,0.1364,0.6723,0.0976,0.0305,0.1862,0.9979,0.4830
4,Final,en,1436,0.2354,0.0689,0.1366,0.6726,0.0977,0.0306,0.1866,1.0000,0.4836
5,Final,pa,3,0.0697,0.0209,0.0439,0.5353,0.0667,0.0000,0.0000,0.0000,0.1108


## Error analysis

In [ ]:
def show(title, cases):
    print(f"\n=== {title}: {len(cases)} cases ===")
    for case in cases[:3]:
        print(json.dumps(case, ensure_ascii=False, indent=1)[:900])


rows = list(zip(test_df["question"], test_df["language"], test_df["final_answer"], test_answers))
show("Wrong output language", [
    {"question": q, "question_language": lang, "answer": a[:300]}
    for q, lang, gold, a in rows if ac.detect_language(a) != lang
])
show("Dosage errors", [
    {"question": q, "missing": sorted(ac.dosages(gold) - ac.dosages(a))[:8], "extra": sorted(ac.dosages(a) - ac.dosages(gold))[:8]}
    for q, lang, gold, a in rows if ac.dosages(gold) and ac.recall(ac.dosages(gold), ac.dosages(a)) < 0.5
])
show("Answers with no gold source", [
    {"question": q, "predicted_sources": s, "gold_sources": json.loads(gold_s)}
    for q, s, gold_s in zip(test_df["question"], final_sources, test_df["sources"]) if not set(s) & set(json.loads(gold_s))
])


=== Wrong output language: 3 cases ===
{
 "question": "ਝੋਨੇ ਦੇ ਨੈਕ ਬਲਾਸਟ ਬਾਰੇ ਜਾਣਕਾਰੀ ਦਿਓ।",
 "question_language": "pa",
 "answer": "The disease is a major problem in the field of agronomic crops. It is a common disease in the field, affecting the crop's growth and yield. It is caused by a variety of pests, including a spp. wasp, a bacterium, and a fungus. The disease is caused by the bacterium spp, which is a bacterial bacterium"
}
{
 "question": "ਮੱਕੀ ਦੇ ਵਿੱਚ ਸੁੰਡੀ ਲਈ ਕੀ ਕਰਨਾ ਚਾਹੀਦਾ",
 "question_language": "pa",
 "answer": "-  ' '  .  ,  &  :  ;  —  –  !    ?  /  a  s  i  n  d  p  t  r  b  c  m  v  l  h  k  w  z  q  x  y  f  g  u  j  o  e  ch  th  if  0  %  Y  X"
}
{
 "question": "ਦਸੰਬਰ ਦੇ ਮਹੀਨੇ ਵਿੱਚ ਕਣਕ ਦੀ ਬਿਜਾਈ ਵਾਸਤੇ ਬਾਰਿਸ਼ ਅਹ ਹੋਣੀ ਲਾਭਦਾਇਕ ਹੈ ਜਾਂ ਨਹੀਂ",
 "question_language": "pa",
 "answer": "-  ' '  .  ,  a  i  n  ;  b  :  s  &  /    –  !  ?  —  p  d  v  t  r  m  c  h  l  u  k  q  w  z  o  f  g  x  y  e  j  ch  th  if  0  %  °  -1"
}

=== Dosage errors: 1070 cases ===
{
 "questio

## Finish

In [ ]:
print(f"Notebook time: {(time.time() - NOTEBOOK_START) / 60:.1f} minutes")
print("Results are in", paths.outputs)
drive.flush_and_unmount()

Notebook time: 11.9 minutes
Results are in /content/drive/MyDrive/agricultural_qa/fast_v1/2effa812/outputs
